# CLONALG — CLONal selection ALGorithm

Clonal selection algorithms are inspired by the adaptive immune system's response to antigens.
B-cells with the highest *affinity* to the antigen are selected, cloned, and subjected to
*somatic hypermutation* — refining the response over successive generations.
CLONALG applies this to continuous optimization: candidates are antibodies, the objective
function is the antigen.

<p align="center">
  <img src="../figures/clonalg_pseudocoge.png" alt="CLONALG pseudocode" width="600"/>
</p>

## Setup

In [1]:
import sys

import numpy as np
import plotly.graph_objects as go

sys.path.insert(0, "../src")
from clonalg import CLONALGConfig, run

## Objective Function

Set `OBJECTIVE` to one of `"Sphere"`, `"Rastrigin"`, or `"Rosenbrock"`.

In [2]:
OBJECTIVE = "Sphere"  # Options: "Sphere", "Rastrigin", "Rosenbrock"


def sphere(x: np.ndarray) -> float:
    """f(x) = Σ xᵢ² — convex, unimodal. Optimum at origin."""
    return float(np.sum(x**2))


def rastrigin(x: np.ndarray) -> float:
    """f(x) = 10n + Σ [xᵢ² - 10cos(2πxᵢ)] — highly multimodal."""
    return float(10 * len(x) + np.sum(x**2 - 10 * np.cos(2 * np.pi * x)))


def rosenbrock(x: np.ndarray) -> float:
    """f(x) = Σ [100(x_{i+1}-xᵢ²)² + (1-xᵢ)²] — narrow curved valley."""
    return float(
        sum(
            100 * (x[i + 1] - x[i] ** 2) ** 2 + (1 - x[i]) ** 2
            for i in range(len(x) - 1)
        )
    )


OBJECTIVES = {
    "Sphere": sphere,
    "Rastrigin": rastrigin,
    "Rosenbrock": rosenbrock,
}

objective_fn = OBJECTIVES[OBJECTIVE]
print(f"Using: {OBJECTIVE} — {objective_fn.__doc__}")

Using: Sphere — f(x) = Σ xᵢ² — convex, unimodal. Optimum at origin.


## Configuration & Execution

In [9]:
config = CLONALGConfig(
    problem_size=3,
    b_lo=-5.0,
    b_up=5.0,
    population_size=200,
    selection_size=20,
    random_cells_num=30,
    clone_rate=50,
    mutation_rate=0.3,
    max_iterations=1000,
)

result = run(objective_fn, config)

print(f"Best solution : {result.best_solution}")
print(f"Best affinity : {result.best_affinity:.8f}")

Best solution : [ 0.15638334 -0.08834823 -0.06616757]
Best affinity : 0.03663931


## Convergence Plot

Interactive chart of the full run — hover to inspect any iteration, scroll to zoom.

In [10]:
best_history = [top5[0][1] for top5 in result.history]
mean_history = [float(np.mean([ind[1] for ind in top5])) for top5 in result.history]
iters = list(range(len(best_history)))

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=iters + iters[::-1],
        y=mean_history + best_history[::-1],
        fill="toself",
        fillcolor="rgba(132, 94, 194, 0.12)",
        line=dict(color="rgba(0,0,0,0)"),
        hoverinfo="skip",
        showlegend=False,
    )
)

fig.add_trace(
    go.Scatter(
        x=iters,
        y=mean_history,
        mode="lines",
        name="Top-5 mean",
        line=dict(color="#845ec2", width=1.5, dash="dot"),
        hovertemplate="Iter %{x}<br>Mean: %{y:.6f}<extra></extra>",
    )
)

fig.add_trace(
    go.Scatter(
        x=iters,
        y=best_history,
        mode="lines",
        name="Best",
        line=dict(color="#00c9a7", width=2.5),
        hovertemplate="Iter %{x}<br>Best: %{y:.8f}<extra></extra>",
    )
)

fig.add_annotation(
    x=len(best_history) - 1,
    y=best_history[-1],
    text=f"  {result.best_affinity:.6f}",
    showarrow=False,
    font=dict(color="#00c9a7", size=11),
    xanchor="left",
)

fig.update_layout(
    title=dict(
        text=f"CLONALG Convergence — {OBJECTIVE} (n={config.problem_size})",
        font=dict(size=15, color="white"),
        x=0.5,
    ),
    paper_bgcolor="#0f1117",
    plot_bgcolor="#161b22",
    font=dict(color="#cccccc"),
    xaxis=dict(
        title="Iteration",
        gridcolor="#1e2530",
        zerolinecolor="#1e2530",
    ),
    yaxis=dict(
        title="Affinity (log scale)",
        type="log",
        gridcolor="#1e2530",
        zerolinecolor="#1e2530",
    ),
    legend=dict(
        bgcolor="#1a1d27",
        bordercolor="#333333",
        borderwidth=1,
    ),
    hovermode="x unified",
    margin=dict(l=60, r=40, t=60, b=50),
)

fig.show()

## References
<a id="references"></a>

1. [Clonal Selection Algorithm — Clever Algorithms](http://www.cleveralgorithms.com/nature-inspired/immune/clonal_selection_algorithm.html)
2. [Clonal Selection Algorithm — Wikipedia](https://en.wikipedia.org/wiki/Clonal_selection_algorithm)
3. de Castro, L. N., & Von Zuben, F. J. (2002). *Learning and optimization using the clonal selection principle*. IEEE Transactions on Evolutionary Computation.